In [20]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.metrics.pairwise import cosine_similarity

import networkx as nx
import matplotlib.pyplot as plt

from utils import _compute_residuals

In [21]:
DATA_PATH = '../dataset/data_andre.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)


# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df['item_id'] = df['item_id'].astype(int) # or .astype(str)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]
print(len(y))

# Calandar-based features
# Ensure datetime
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

# Base calendar parts
df["day_of_week"]  = df[DATE_COL].dt.dayofweek.astype(int)
df["day_of_month"] = df[DATE_COL].dt.day.astype(int)         
df["moy"]          = df[DATE_COL].dt.month.astype(int)-1      
df["doy"]          = df[DATE_COL].dt.dayofyear.astype(int)-1   
df["is_weekend"] = (
    (df[DATE_COL].dt.dayofweek == 5) |
    (df[DATE_COL].dt.dayofweek == 6)
).astype(int)

df = df.sort_values(["item_id", DATE_COL]).reset_index(drop=True)

df

Loading data from ../dataset/data_andre.feather...
1082371


,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,promo_type_CLCP,promo_value_CLCP,promo_type_LFPE,promo_value_LFPE,store_id,day_of_week,day_of_month,moy,doy,is_weekend
0,2021-01-23,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0.0,0,0.0,6269,5,23,0,22,1
1,2021-01-24,27,14,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0.0,0,0.0,6269,6,24,0,23,1
2,2021-01-25,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0.0,0,0.0,6269,0,25,0,24,0
3,2021-01-26,27,5,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0.0,0,0.0,6269,1,26,0,25,0
4,2021-01-27,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0.0,0,0.0,6269,2,27,0,26,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1082366,2023-02-18,900087600,28,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0,0.0,0,0.0,6269,5,18,1,48,1
1082367,2023-02-19,900087600,46,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0,0.0,0,0.0,6269,6,19,1,49,1
1082368,2023-02-20,900087600,28,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0,0.0,0,0.0,6269,0,20,1,50,0
1082369,2023-02-21,900087600,31,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0,0.0,0,0.0,6269,1,21,1,51,0


In [22]:
def build_residual_correlation_graph(
    df: pd.DataFrame,
    node_col: str = "item_id",
    date_col: str = "date",
    value_col: str = "value",
    trend_window: int = 7,
    min_overlap: int = 10,
    corr_method: str = "pearson",
    corr_threshold: float = 0.2,
    absolute_corr: bool = True,
):
    """
    Build a residual-correlation graph from daily node evolution.
    """
    data, residuals_wide = _compute_residuals(
        df=df,
        node_col=node_col,
        date_col=date_col,
        value_col=value_col,
        trend_window=trend_window,
    )

    corr_matrix = residuals_wide.corr(method=corr_method)

    not_na_matrix = residuals_wide.notna().astype(int)
    overlap = not_na_matrix.T @ not_na_matrix

    G = nx.Graph()

    node_stats = (
        data.groupby(node_col)
        .agg(
            n_obs=(value_col, "count"),
            mean_value=(value_col, "mean"),
            std_value=(value_col, "std"),
            mean_residual=("residual", "mean"),
            std_residual=("residual", "std"),
        )
        .reset_index()
    )

    for _, row in node_stats.iterrows():
        G.add_node(
            row[node_col],
            n_obs=int(row["n_obs"]),
            mean_value=float(row["mean_value"]) if pd.notna(row["mean_value"]) else 0.0,
            std_value=float(row["std_value"]) if pd.notna(row["std_value"]) else 0.0,
            mean_residual=float(row["mean_residual"]) if pd.notna(row["mean_residual"]) else 0.0,
            std_residual=float(row["std_residual"]) if pd.notna(row["std_residual"]) else 0.0,
        )

    nodes = corr_matrix.columns.tolist()
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            n1, n2 = nodes[i], nodes[j]
            corr = corr_matrix.loc[n1, n2]
            n_common = overlap.loc[n1, n2]

            if pd.isna(corr) or n_common < min_overlap:
                continue

            keep = abs(corr) >= corr_threshold if absolute_corr else corr >= corr_threshold

            if keep:
                G.add_edge(
                    n1,
                    n2,
                    weight=float(corr),
                    abs_weight=float(abs(corr)),
                    n_common=int(n_common),
                    sign="positive" if corr >= 0 else "negative",
                )

    return G, corr_matrix, residuals_wide

In [23]:
G, corr_mat, residuals_df = build_residual_correlation_graph(df, corr_threshold=0.8)
print(f"Graph nodes: {G.number_of_nodes()}, Graph edges: {G.number_of_edges()}")


Graph nodes: 1427, Graph edges: 2


In [25]:
for edge in G.edges(data=True):
    #print ids of the edge
    df_edge = df[df['item_id'].isin([edge[0], edge[1]])]
    print(df_edge)
    
    

             date  item_id  value       cat_label            sdep_label  \
242960 2021-01-23    26002     25  sprd btr mrgrn  pos subd dairy other   
242961 2021-01-24    26002     28  sprd btr mrgrn  pos subd dairy other   
242962 2021-01-25    26002     19  sprd btr mrgrn  pos subd dairy other   
242963 2021-01-26    26002     20  sprd btr mrgrn  pos subd dairy other   
242964 2021-01-27    26002     31  sprd btr mrgrn  pos subd dairy other   
...           ...      ...    ...             ...                   ...   
245238 2023-02-18    26012      4  sprd btr mrgrn  pos subd dairy other   
245239 2023-02-19    26012     10  sprd btr mrgrn  pos subd dairy other   
245240 2023-02-20    26012     11  sprd btr mrgrn  pos subd dairy other   
245241 2023-02-21    26012      3  sprd btr mrgrn  pos subd dairy other   
245242 2023-02-22    26012      5  sprd btr mrgrn  pos subd dairy other   

             dep_label    dmn_label  promo_type_FRPG  promo_value_FRPG  \
242960  pos dept dairy  o